This notebook: performs imputed methylome ~ phecode phenotype associations (MWAS)
- creates cases / control covariates+pheno matrices to be used in MWAS
- run MWAS for each phenotype in batches using SAK
- needs (1) predictions matrix (2) cases matrix (3) controls matrix (4) mwas_03_assoc.sh (5) mwas_03.1_assoc.py (6) mwas_func.py (7) phecode_map.csv 
- Date: 16.02.2026


## Setup

In [1]:
%%bash
pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 35.6 MB/s eta 0:00:00a 0:00:01


In [1]:
import os
import glob
import pandas as pd
from pandas.core.common import flatten
import re
import numpy as np
import seaborn as sns
from tqdm import tqdm
import time
import sys
# import statsmodels.api as sm
from scipy import stats
from scipy.stats import zscore
from scipy.stats import pearsonr

## 1. Prepare ICD-10 disease phenotypes

### 1.1 load long ICD-10 file

In [3]:
!dx download -f ismail/GxRF/1_Initial_Dataset/incidence_cases.noncancer.timing_and_disease_status.melted_dataframe.final.tsv
!dx download -f ismail/GxRF/1_Initial_Dataset/prevalent_cases.noncancer.timing_and_disease_status.melted_dataframe.final.tsv 
!dx download -f ismail/GxRF/1_Initial_Dataset/controls.censoring_time.tsv

[===========================================================>] Completed 391,575,248 of 391,575,248 bytes (100%) /opt/notebooks/incidence_cases.noncancer.timing_and_disease_status.melted_dataframe.final.tsvv
[===========================================================>] Completed 346,850,960 of 346,850,960 bytes (100%) /opt/notebooks/prevalent_cases.noncancer.timing_and_disease_status.melted_dataframe.final.tsvv
[===========================================================>] Completed 5,697,828 of 5,697,828 bytes (100%) /opt/notebooks/controls.censoring_time.tsvv


In [4]:
# prevelant 
prev   = pd.read_csv('prevalent_cases.noncancer.timing_and_disease_status.melted_dataframe.final.tsv', sep="\t")
#incident 
incid  = pd.read_csv('incidence_cases.noncancer.timing_and_disease_status.melted_dataframe.final.tsv', sep="\t")
# both 
icd = pd.concat([incid, prev], axis=0, ignore_index=True)
print(icd.shape)
icd.head()

(10168582, 10)


,eid_variable,eid,trait_code,date,Recruitment,time,icd_code,letter,num,CombinedTrait
0,2537387_A00-A09_A00,2537387,A00,2010-12-22,2009-10-01,447,A00,A,0,A00-A09
1,5887626_A00-A09_A00,5887626,A00,2012-09-05,2010-03-01,919,A00,A,0,A00-A09
2,5465066_A00-A09_A00,5465066,A00,2016-09-06,2010-01-16,2425,A00,A,0,A00-A09
3,3074422_A00-A09_A00,3074422,A00,2014-02-13,2008-05-22,2093,A00,A,0,A00-A09
4,4156175_A00-A09_A01,4156175,A01,2012-07-01,2007-06-11,1847,A01,A,1,A00-A09


In [5]:
# traits with >150 cases
ncas_min = 150
codes_n = icd.groupby('trait_code').size()
codes_n = pd.DataFrame(codes_n[codes_n > ncas_min])
codes_n.index.names = ['code']
codes_n.columns = ['n']
print(codes_n.shape[0])
codes_n.head()

550


,n
code,
A01,180
A02,786
A03,208
A04,17476
A05,394


### 1.2 add phecodes
- see here: https://academic.oup.com/bioinformatics/article/39/11/btad655/7335839


In [6]:
!git clone https://github.com/PheWAS/PhecodeX

Cloning into 'PhecodeX'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 84 (delta 41), reused 44 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 1.04 MiB | 18.93 MiB/s, done.
Resolving deltas: 100% (41/41), done.


#### 1.2.1 ICD10 <-> phecode mapping table

In [7]:
icd10_labels = pd.read_csv('PhecodeX/phecodeX_R_labels.csv').rename(columns = {'phenotype': 'phecode'}).loc[:, ['phecode', 'description', 'group', 'color'] ]
icd10_labels['phecode'] = icd10_labels['phecode'].astype(str)

In [8]:
icd10_map = pd.read_csv('PhecodeX/phecodeX_R_map.csv')
icd10_map = icd10_map.loc[ icd10_map['vocabulary_id'] == 'ICD10CM', ['code','phecode'] ]
icd10_map['phecode'] = icd10_map['phecode'].astype(str)
icd10_map = pd.merge(icd10_map, icd10_labels, on='phecode', how='inner').set_index('code')
icd10_map.head()

,phecode,description,group,color
code,,,,
D53,BI_160,Deficiency anemias,Blood/Immune,#FFFF00
D53.2,BI_160,Deficiency anemias,Blood/Immune,#FFFF00
D53.8,BI_160,Deficiency anemias,Blood/Immune,#FFFF00
D53.9,BI_160,Deficiency anemias,Blood/Immune,#FFFF00
D50,BI_160.1,Iron deficiency anemia,Blood/Immune,#FFFF00


In [9]:
## phecodes in UKB ICD-10 codes
phecode_tbl = codes_n.join(icd10_map, how='inner')
print(phecode_tbl.shape[0])
phecode_tbl.head()

602


,n,phecode,description,group,color
code,,,,,
A01,180,GI_526,Intestinal infection,Gastrointestinal,#FFD700
A01,180,ID_001,Salmonella,Infections,#FF0000
A01,180,ID_089.1,Bacterial infections,Infections,#FF0000
A02,786,GI_526.1,Bacterial enteritis,Gastrointestinal,#FFD700
A02,786,ID_001,Salmonella,Infections,#FF0000


#### 1.2.2 add phecode to phenotypes table

In [179]:
icd_phe = icd.set_index('trait_code').join(phecode_tbl, how='inner').reset_index()
icd_phe = icd_phe[ ['eid', 'date', 'trait_code', 'phecode', 'description','group', 'color'] ] 
print(icd_phe.shape[0])
icd_phe.head()

10785880


,eid,date,trait_code,phecode,description,group,color
0,4156175,2012-07-01,A01,GI_526,Intestinal infection,Gastrointestinal,#FFD700
1,4156175,2012-07-01,A01,ID_001,Salmonella,Infections,#FF0000
2,4156175,2012-07-01,A01,ID_089.1,Bacterial infections,Infections,#FF0000
3,5746251,2014-07-01,A01,GI_526,Intestinal infection,Gastrointestinal,#FFD700
4,5746251,2014-07-01,A01,ID_001,Salmonella,Infections,#FF0000


In [82]:
phecodes_n = icd_phe.groupby('phecode').size()
phecodes_n

phecode
BI_160          301
BI_160.1      68208
BI_160.2       3828
BI_160.22      1175
BI_162         1791
              ...  
SO_394        26716
SO_395        24376
SO_396        62052
SO_397        19354
SS_840.8     108764
Length: 445, dtype: int64

## 2 make covariates file

In [14]:
!dx download -f vasilis/data/pheno_all_unfiltered.csv

[===========================================================>] Completed 229,829,671 of 229,829,671 bytes (100%) /opt/notebooks/pheno_all_unfiltered.csvv


In [15]:
raw = pd.read_csv('pheno_all_unfiltered.csv', sep=",")

/tmp/ipykernel_68/3378516098.py:1: DtypeWarning: Columns (63,64,73,74,75,76,77,78,80,81,82,83,84,85,86,87,88,100) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv('pheno_all_unfiltered.csv', sep=",")


In [16]:
pcs = [f"p22009_a{i}" for i in range(1, 21)]
cols = ['eid', 'p31', 'p34', 'p52', 'p40007_i0', 'p22000'] + pcs
cov0 = raw.loc[:,cols]
cov0.head()

,eid,p31,p34,p52,p40007_i0,p22000,p22009_a1,p22009_a2,p22009_a3,p22009_a4,...,p22009_a11,p22009_a12,p22009_a13,p22009_a14,p22009_a15,p22009_a16,p22009_a17,p22009_a18,p22009_a19,p22009_a20
0,1000015,1,1940,11,NaN,50.0,-11.09950,2.14877,-0.949851,1.180510,...,1.00230,1.574560,-2.916850,2.175070,1.907590,4.51903,-2.286550,3.96858,-0.788353,1.930670
1,1000053,0,1968,2,NaN,19.0,-12.19880,2.87228,1.880270,0.808897,...,2.34417,0.694553,-0.654478,-0.361432,-1.871140,-6.93788,0.730852,7.18108,3.132090,-1.143680
2,1000132,0,1951,3,NaN,44.0,-11.85840,3.91488,-0.775548,0.268560,...,8.55607,-3.597410,0.352823,1.360030,1.547080,9.72470,-6.842110,4.99451,3.512240,-3.187870
3,1000148,0,1956,11,54.3,38.0,-12.33350,3.50807,-1.238640,2.310040,...,4.77963,0.918477,1.782720,6.454620,3.876280,16.26520,-0.318199,-2.53354,-2.746670,3.162380
4,1000163,1,1968,9,NaN,94.0,-9.72894,5.65563,-0.851135,-3.722650,...,-3.20133,-0.485291,-1.284440,-1.847220,0.604082,-1.67233,-0.158524,2.69347,0.116500,0.198556


In [53]:
# make dummy DOB
cov0['dob'] = pd.to_datetime(dict(year=cov0['p34'], month=cov0['p52'], day=1))
# make censoring age
cov0['censoring_date'] = pd.to_datetime('2023-3-31')
cov0['censoring_age']  = (cov0['censoring_date'] - cov0['dob']).dt.days / 365.25
cov0['censoring_age']  = cov0['censoring_age'].round(1)
# censoring age or death age
cov0['censoring_age']  = cov0[['censoring_age', 'p40007_i0']].min(axis=1)
# select
pcs_old = [f"p22009_a{i}" for i in range(1, 21)] 
pcs_new = [f"PC{i}" for i in range(1, 21)] 
cov_cols = ['eid', 'sex', 'batch', 'dob', 'censoring_age'] + pcs_new
cov1 = cov0.rename(columns={'p31': 'sex', 'p22000': 'batch'}).rename(columns=dict(zip(pcs_old, pcs_new))).loc[ :, cov_cols ]
cov1['batch'] = pd.to_numeric(cov1['batch'], errors='coerce').astype('Int64')
cov1.head()

,eid,sex,batch,dob,censoring_age,PC1,PC2,PC3,PC4,PC5,...,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20
0,1000015,1,50,1940-11-01,82.4,-11.09950,2.14877,-0.949851,1.180510,-4.60573,...,1.00230,1.574560,-2.916850,2.175070,1.907590,4.51903,-2.286550,3.96858,-0.788353,1.930670
1,1000053,0,19,1968-02-01,55.2,-12.19880,2.87228,1.880270,0.808897,2.96457,...,2.34417,0.694553,-0.654478,-0.361432,-1.871140,-6.93788,0.730852,7.18108,3.132090,-1.143680
2,1000132,0,44,1951-03-01,72.1,-11.85840,3.91488,-0.775548,0.268560,-10.28480,...,8.55607,-3.597410,0.352823,1.360030,1.547080,9.72470,-6.842110,4.99451,3.512240,-3.187870
3,1000148,0,38,1956-11-01,54.3,-12.33350,3.50807,-1.238640,2.310040,9.26804,...,4.77963,0.918477,1.782720,6.454620,3.876280,16.26520,-0.318199,-2.53354,-2.746670,3.162380
4,1000163,1,94,1968-09-01,54.6,-9.72894,5.65563,-0.851135,-3.722650,-3.76724,...,-3.20133,-0.485291,-1.284440,-1.847220,0.604082,-1.67233,-0.158524,2.69347,0.116500,0.198556


In [54]:
!dx download -f vasilis/data/ukb_wb_del_cov_e4_dem_alz_phe.txt

[===========================================================>] Completed 87,320,990 of 87,320,990 bytes (100%) /opt/notebooks/ukb_wb_del_cov_e4_dem_alz_phe.txtt


In [180]:
## add APOE e4 status
extra = pd.read_csv('ukb_wb_del_cov_e4_dem_alz_phe.txt', sep=" ")
extra = extra[ ['IID','e4_status'] ].set_index('IID')
extra.index.name = 'eid'
cov2 = cov1.set_index('eid').join(extra, how='left',rsuffix='_1').reset_index()
cov2.head()

,eid,sex,batch,dob,censoring_age,PC1,PC2,PC3,PC4,PC5,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,e4_status
0,1000015,1,50,1940-11-01,82.4,-11.09950,2.14877,-0.949851,1.180510,-4.60573,...,1.574560,-2.916850,2.175070,1.907590,4.51903,-2.286550,3.96858,-0.788353,1.930670,1.0
1,1000053,0,19,1968-02-01,55.2,-12.19880,2.87228,1.880270,0.808897,2.96457,...,0.694553,-0.654478,-0.361432,-1.871140,-6.93788,0.730852,7.18108,3.132090,-1.143680,2.0
2,1000132,0,44,1951-03-01,72.1,-11.85840,3.91488,-0.775548,0.268560,-10.28480,...,-3.597410,0.352823,1.360030,1.547080,9.72470,-6.842110,4.99451,3.512240,-3.187870,0.0
3,1000148,0,38,1956-11-01,54.3,-12.33350,3.50807,-1.238640,2.310040,9.26804,...,0.918477,1.782720,6.454620,3.876280,16.26520,-0.318199,-2.53354,-2.746670,3.162380,0.0
4,1000163,1,94,1968-09-01,54.6,-9.72894,5.65563,-0.851135,-3.722650,-3.76724,...,-0.485291,-1.284440,-1.847220,0.604082,-1.67233,-0.158524,2.69347,0.116500,0.198556,0.0


## 3. Save required files

In [181]:
icd_phe.to_csv('icd.phe', index=False)
cov2.to_csv('cov.phe', index=False)
phecode_tbl.to_csv('phecode_map.csv', index=True)

In [182]:
!dx upload --brief icd.phe --dest vasilis/data/ebb/phenotypes/
!dx upload --brief cov.phe --dest vasilis/data/ebb/phenotypes/
!dx upload --brief phecode_map.csv --dest vasilis/data/ebb/phenotypes/

file-J69Q270JZB77Qy1YfB5Kp3y4
file-J69Q280JZB7ByggGzj10q10p
file-J69Q28jJZB7Kkkb5Ffkg44Z6


## 4. Run methylome wide associations 
- in batches - use SAK 
- results in: `vasilis/data/ebb/results/[phecode]__batch*.csv`


In [2]:
%%bash
dx download -f vasilis/SAK_scripts/mwas*

In [72]:
%%bash
dx upload --brief mwas_03_assoc.sh --dest vasilis/SAK_scripts/
dx upload --brief mwas_03.1_assoc.py --dest vasilis/SAK_scripts/
dx upload --brief mwas_func.py --dest vasilis/SAK_scripts/

file-J6B9kq0JZB7K177bqzZ1g9q1
file-J6B9kq8JZB7K177bqzZ1g9q3
file-J6B9kqQJZB77QPQBP2y45GP8


In [ ]:
### inputs needed: 
## passed by -iin: batch*_predict.txt
## passed by -iin: icd.phe 
## passed by -iin: cov.phe
## passed by -iin: phecode_map.csv
## passed by -iin: mwas_func.py
## passed by -iin: mwas_03.1_assoc.py
## passed by -iin: mwas_03_assoc.sh
## passed by -icmd: $1 batch

In [ ]:
%%bash

icd="vasilis/data/ebb/phenotypes/icd.phe"
cov="vasilis/data/ebb/phenotypes/cov.phe"
phemap="vasilis/data/ebb/phenotypes/phecode_map.csv"
predictdir="vasilis/data/ebb/scores"
dest="vasilis/data/ebb/results/"

for batch in {3..50}; do
    dx run swiss-army-knife \
        -iin="vasilis/SAK_scripts/mwas_03_assoc.sh" \
        -iin="vasilis/SAK_scripts/mwas_03.1_assoc.py" \
        -iin="vasilis/SAK_scripts/mwas_func.py" \
        -iin="${predictdir}/batch${batch}_predict.txt" \
        -iin="${icd}" \
        -iin="${cov}" \
        -iin="${phemap}" \
        -icmd="sh mwas_03_assoc.sh ${batch}" \
        --tag="as_${batch}" \
        --instance-type "mem1_ssd1_v2_x36" \
        --destination="${dest}" \
        --brief --yes --priority high
done


## 5. sensitivity MWAS
- run extra sensitivity MWAS for selected phenotype eg. APOE-e4 adjusted for delirium
- needs mwas_04.0_assoc_sens.sh; mwas_04.1_assoc_sens.py; phecode_map_sens.csv
- results in: `vasilis/data/ebb/results/[phecode]__batch*_[sens].csv`

### 5.1 *APOE-e4* adjusted for delirium 

In [2]:
phecodes = pd.read_csv('/mnt/project/vasilis/data/ebb/phenotypes/phecode_map.csv')

In [9]:
descriptions = ['Delirium','Dementias', 'Alzheimer\'s disease']
phecode_map_sens = phecodes.loc[ phecodes['description'].isin(descriptions) ]
phecode_map_sens.to_csv('phecode_map_sens.csv', index=False)

In [10]:
!dx upload --brief phecode_map_sens.csv --dest vasilis/data/ebb/phenotypes/

file-J6G4VJ8JZB73GBkpQqz5x35f


In [12]:
!dx upload --brief mwas_04.0_assoc_sens.sh --dest vasilis/SAK_scripts/
!dx upload --brief mwas_04.1_assoc_sens.py --dest vasilis/SAK_scripts/

file-J6FPY60JZB79jvyZg2y5vjjv
file-J6FPY6QJZB79jvyZg2y5vk36


In [12]:
%%bash

icd="vasilis/data/ebb/phenotypes/icd.phe"
cov="vasilis/data/ebb/phenotypes/cov.phe"
phemap="vasilis/data/ebb/phenotypes/phecode_map_sens.csv"
predictdir="vasilis/data/ebb/scores"
dest="vasilis/data/ebb/results/"

for batch in {2..50}; do
    dx run swiss-army-knife \
        -iin="vasilis/SAK_scripts/mwas_04.0_assoc_sens.sh" \
        -iin="vasilis/SAK_scripts/mwas_04.1_assoc_sens.py" \
        -iin="vasilis/SAK_scripts/mwas_func.py" \
        -iin="${predictdir}/batch${batch}_predict.txt" \
        -iin="${icd}" \
        -iin="${cov}" \
        -iin="${phemap}" \
        -icmd="sh mwas_04.0_assoc_sens.sh ${batch}" \
        --tag="as_${batch}" \
        --instance-type "mem1_ssd1_v2_x36" \
        --destination="${dest}" \
        --brief --yes --priority high
done


job-J6G4b90JZB74YBxVpzz6KJf3
job-J6G4b98JZB7GbfY9VpYqz5yp
job-J6G4b9QJZB76q6ZzVBK4Kv17
job-J6G4bB0JZB74PZKQY2yFPFGG
job-J6G4bB8JZB7GbfY9VpYqz5z5
job-J6G4bBQJZB74PZKQY2yFPFGQ
job-J6G4bBjJZB72J9q8qQJPgvJg
job-J6G4bF0JZB7KXF04JBB0B4vJ
job-J6G4bF8JZB7GPQb6Xf0bbGBg
job-J6G4bFQJZB7GPQb6Xf0bbGBp
job-J6G4bFjJZB7GbfY9VpYqz5z9
job-J6G4bG8JZB74YBxVpzz6KJfj
job-J6G4bGQJZB74YBxVpzz6KJfp
job-J6G4bGjJZB7GbfY9VpYqz5zF
job-J6G4bJ0JZB74YBxVpzz6KJfv
job-J6G4bJ8JZB76q6ZzVBK4Kv1X
job-J6G4bJQJZB74YBxVpzz6KJfz
job-J6G4bJjJZB7GbfY9VpYqz5zV
job-J6G4bK0JZB7GbfY9VpYqz5zY
job-J6G4bK8JZB74PZKQY2yFPFJ2
job-J6G4bKQJZB7GbfY9VpYqz5zf
job-J6G4bKjJZB74PZKQY2yFPFJ4
job-J6G4bP0JZB76q6ZzVBK4Kv27
job-J6G4bPQJZB74PZKQY2yFPFJ8
job-J6G4bPjJZB74PZKQY2yFPFJB
job-J6G4bQ0JZB76q6ZzVBK4Kv2F
job-J6G4bQ8JZB7GbfY9VpYqz5zv
job-J6G4bQQJZB76q6ZzVBK4Kv2J
job-J6G4bQjJZB74PZKQY2yFPFJG
job-J6G4bV0JZB7GbfY9VpYqz5zy
job-J6G4bV8JZB73GYZb3YxG7FPY
job-J6G4bVQJZB7GbfY9VpYqz600
job-J6G4bVjJZB74YBxVpzz6KJgJ
job-J6G4bX0JZB74YBxVpzz6KJgP
job-J6G4bX8JZB